<a href="https://colab.research.google.com/github/kylejgillett/sounderpy/blob/main/colab_tutorials/sounderpy_forecast_models.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Plotting Forecast Soundings with SounderPy

Get a forecast sounding from a weather model (GFS, RAP, HRRR, or RRFS) through BUFKIT and plot it, in a few lines of code. You'll learn to:

1. install and import SounderPy
2. get the latest forecast, or an archived one, with `spy.get_bufkit_data()`
3. look inside the data
4. plot a sounding and a hodograph, and change how they look
5. save the figure and the data

📖 [SounderPy docs](https://sounderpy.app/docs/) · [BUFKIT data reference](https://sounderpy.app/docs/reference/gettingdata.html#model-forecast-data-bufkit-profiles)

---
## Step 1 · Install and import SounderPy

Run a cell by clicking ▶ at its left, or press **Shift + Enter**. A ✅ next to the cell means it ran.

The first line installs SounderPy into this Colab session (once per session). The second imports it as `spy`, the short name used in every SounderPy example. [Getting started](https://sounderpy.app/docs/user_guide/getting_started.html)

In [ ]:
%pip install -q "sounderpy>=3.3.0"

import sounderpy as spy

---
## Step 2 · Get the data

BUFKIT files hold model forecast soundings for a fixed list of sites, mostly airports and NWS offices. SounderPy can get forecasts **only at those sites**. [Find a site](https://sounderpy.app/docs/reference/gettingdata.html#available-bufkit-sites)

`spy.get_bufkit_data()` downloads one forecast sounding and returns it ready to plot:

```python
model_data = spy.get_bufkit_data(model, station, forecast_hour)
```

* `model`: for the latest run, `"hrrr"`, `"rap"`, `"gfs"`, or `"rrfs"`. [Available models](https://sounderpy.app/docs/reference/gettingdata.html#available-models)
* `station`: a BUFKIT site ID, written as BUFKIT lists it: some start with K (`"KMOP"`, Mount Pleasant, MI), others don't (`"OUN"`).
* `forecast_hour`: hours after the model run; `0` is the run time itself.
* `hush=True` skips the printed summary of the sounding's parameters.

This gets the latest HRRR run's 6-hour forecast for Mount Pleasant:

In [ ]:
model_data = spy.get_bufkit_data("hrrr", "KMOP", 6, hush=True)

### An archived forecast

Add the model run's date and hour (UTC) to get an older forecast. Archived runs also include the NAM and NAM nest (`"nam"`, `"namnest"`). This is the 12Z GFS from August 5, 2023, at forecast hour 6:

In [ ]:
archived_data = spy.get_bufkit_data("gfs", "KMOP", 6, "2023", "08", "05", "12", hush=True)

---
## Step 3 · Look inside the data

`model_data` is a Python dictionary SounderPy calls `clean_data`. Every data source returns the same layout, so everything below works the same for observed, model, and your own data:

| Key | Holds |
|---|---|
| `p`, `z` | pressure (hPa) and height (m above sea level) |
| `T`, `Td` | temperature and dewpoint (°C) |
| `u`, `v` | wind components (kt) |
| `site_info` | where and when: site ID, name, lat/lon, elevation, valid time |

The values carry their units (they are [MetPy](https://unidata.github.io/MetPy/) quantities). [The `clean_data` scheme](https://sounderpy.app/docs/reference/clean_data_scheme.html)

In [ ]:
print(model_data["site_info"])

# the lowest five levels
print("Pressure:   ", model_data["p"][:5])
print("Temperature:", model_data["T"][:5])
print("Dewpoint:   ", model_data["Td"][:5])

---
## Step 4 · Plot it

`spy.build_sounding()` draws the full sounding: the skew-T, hodograph, map, and a table of parameters. Right-click the figure to save or open it.

In [ ]:
spy.build_sounding(model_data)

The map shows the U.S. radar mosaic at the forecast's valid time. That mosaic only goes back about a month, so for the archived forecast, `radar="single"` uses the nearest NEXRAD radar instead. For a forecast valid in the future, the map shows the latest radar available. [Maps & radar](https://sounderpy.app/docs/tutorials/maps_radar.html)

In [ ]:
spy.build_sounding(archived_data, radar="single")

### Change how it looks

Settings are *keyword arguments* added after the data. A few to try:

| Setting | Does |
|---|---|
| `dark_mode=True` | a dark figure |
| `color_blind=True` | a blue dewpoint line instead of green |
| `storm_motion="left_moving"` | a different [storm motion](https://sounderpy.app/docs/reference/plotsettings.html#storm-motion-logic): `"right_moving"` (default), `"left_moving"`, `"mean_wind"`, or your own `[direction, speed]` like `[250, 45]` |
| `special_parcels="simple"` | only the basic parcels: faster ([parcels](https://sounderpy.app/docs/tutorials/parcel_settings.html)) |
| `show_theta=True` | potential temperature profiles in place of the CAPE panel |
| `mode="winter"` | a [winter sounding](https://sounderpy.app/docs/tutorials/winter_soundings.html): precipitation type and snow growth |
| `radar=None`, `map_zoom=0` | no radar, or no map at all |

Change the values below and run the cell again. [All sounding settings](https://sounderpy.app/docs/reference/plottingdata.html#soundings) · [Plot appearance](https://sounderpy.app/docs/tutorials/plot_appearance.html)

In [ ]:
spy.build_sounding(model_data, dark_mode=True, color_blind=True, storm_motion="mean_wind")

### Just the hodograph

`spy.build_hodograph()` takes the same data and most of the same settings. `sr_hodo=True` draws it storm relative. [Hodograph options](https://sounderpy.app/docs/tutorials/hodograph_options.html)

In [ ]:
spy.build_hodograph(model_data, sr_hodo=True)

---
## Step 5 · Save your work

`save=True` writes the figure to a PNG instead of showing it, and `spy.to_file()` saves the data as a CSV, a SHARPpy file, or a CM1 `input_sounding`. Find the files in the 📁 panel on the left (right-click a file to download it). [Exporting data](https://sounderpy.app/docs/tutorials/exporting_data.html)

In [ ]:
spy.build_sounding(model_data, save=True, filename="kmop_hrrr_f06_sounding.png")

spy.to_file("csv", model_data, filename="kmop_hrrr_f06.csv")

---
## Keep going

* [Beginner's cookbook](https://colab.research.google.com/github/kylejgillett/sounderpy/blob/main/colab_tutorials/beginners_sounderpy_cookbook.ipynb): every data source, parameters, composites, and the command line
* [Tutorials](https://sounderpy.app/docs/tutorials/index.html): [plot appearance](https://sounderpy.app/docs/tutorials/plot_appearance.html), [maps & radar](https://sounderpy.app/docs/tutorials/maps_radar.html), [parcels](https://sounderpy.app/docs/tutorials/parcel_settings.html), [hodographs](https://sounderpy.app/docs/tutorials/hodograph_options.html), [winter soundings](https://sounderpy.app/docs/tutorials/winter_soundings.html)
* [Plot gallery](https://sounderpy.app/docs/gallery/examplegallery.html): figures with the code that made them
* [Getting data](https://sounderpy.app/docs/reference/gettingdata.html): every data source SounderPy can retrieve

**Learning Python for meteorology**: [Unidata Python training](https://unidata.github.io/python-training/) · [MetPy Mondays (YouTube)](https://www.youtube.com/playlist?list=PLQut5OXpV-0ir4IdllSt1iEZKTwFBa7kO) · [Python for beginners](https://www.python.org/about/gettingstarted/)

Questions? See the [SounderPy community page](https://sounderpy.app/docs/user_guide/community.html).

*(C) Kyle J Gillett, University of North Dakota*